In [20]:
import folium
from folium import plugins
import psycopg2
import json

# Connect to PostGIS
conn = psycopg2.connect(
    dbname="utahdaminundationprofiles_aug9_2025",
    user="admin",
    password="admin",
    host="localhost",
    port=5432
)
cur = conn.cursor()

# Red means more affecting power plants, blue is less affecting 

Features
Connects to your PostGIS DB.

Retrieves the top 50 largest dams from utah_dams (by Shape__Area), including those with 0 intersecting power plants.

Colors dams based on the number of intersecting power plants:

- Gray for 0

- Orange for 1

- Red for 2 or more

- Adds popups with dam name, area, and number of affected power plants.

- Adds a colormap legend.

- Saves the result to dams_colored_by_impact.html.



📍 Map file: dams_colored_by_impact.html

🔵 Blueish-gray dams: no power plants affected

🟠 Orange: one plant

🔴 Red: two or more plants

ℹ️ Hover tooltips + popups with details

🔲 Fullscreen toggle + colorbar legend

In [15]:
import folium
from folium import plugins
import psycopg2
import json
from branca.colormap import StepColormap
import branca.colormap as cm

# ----------------------------
# Connect to PostGIS
# ----------------------------
conn = psycopg2.connect(
    dbname="utahdaminundationprofiles_aug9_2025",
    user="admin",
    password="admin",
    host="localhost",
    port=5432
)
cur = conn.cursor()

# ----------------------------
# Create folium map
# ----------------------------
m = folium.Map(location=[40.7607, -111.8939], zoom_start=7)

# ----------------------------
# Query top 50 largest dams (including 0-intersection)
# ----------------------------
cur.execute("""
    WITH plant_geoms AS (
        SELECT objectid, id, beds, trauma, ST_SetSRID(ST_MakePoint(longitude, latitude), 4326) AS geom 
        FROM hospitals
        WHERE "latitude" IS NOT NULL AND "longitude" IS NOT NULL
    ),
    dams_with_capacity AS (
        SELECT
            d.objectid,
            d.damnumber,
            d.name,
            ST_AsGeoJSON(d.geom) AS geojson,
            SUM(p.beds) AS total_beds,
            COUNT(p.id) AS num,
            MAX(p.trauma) AS trauma
        FROM utah_dam_inundation_zones d
        JOIN plant_geoms p
          ON ST_Intersects(d.geom, p.geom)
        GROUP BY d.objectid, d.name, d.shape_area, d.geom
    )
    SELECT *
    FROM dams_with_capacity
""")

results = cur.fetchall()

min_intersect_count = min(int(r[4]) for r in results)
max_intersect_count = max(int(r[4]) for r in results)
print(max_intersect_count)
colormap = cm.linear.Reds_09.scale(min_intersect_count, max_intersect_count)

# ----------------------------
# Define colormap
# ----------------------------
#colormap = StepColormap(
    #colors=['#d3d3d3','#FFFF00', '#feb24c', '#f03b20'],  # gray, orange, red
    #index=[1, 2, 3, 4, 5],
#    vmin=min_intersect_count,
#    vmax=max_intersect_count
#)
colormap.caption = "Total number of beds in hospitals"
colormap.add_to(m)

# ----------------------------
# Add each dam to map

dam_layer = folium.FeatureGroup(name="Inundation Zones")

for dam_id, dam_num, name, geom_json, beds, num, trauma in results:
    geojson = json.loads(geom_json)
    popup_html = f"""
        <b>Dam:</b> {name or 'Unknown'}<br>
        <b>Dam Number:</b> {dam_num or 'Unknown'}<br>
        <b>Total beds:</b> {beds} <br>
        <b>Total number of hospitals:</b> {num} <br>
        <b>Trauma Level:</b> {trauma}
    """
    folium.GeoJson(
        data=geojson,
        style_function=lambda feature, var=beds: {
            "fillColor": colormap(beds),
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.6
        },
        highlight_function=lambda x: {"weight": 2, "color": "yellow"},
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=name
    ).add_to(dam_layer)

dam_layer.add_to(m)

folium.LayerControl().add_to(m)

# ----------------------------
# Cleanup
# ----------------------------
cur.close()
conn.close()




395


In [16]:
# ----------------------------
# Add fullscreen button and save
# ----------------------------
plugins.Fullscreen().add_to(m)
m.save("./html/dams_hospitals.html")
m

In [7]:
import pandas as pd
column_names = ["dam_id", "name", "area_m2", "geom_json", "total_oper_cap"]
df = pd.DataFrame(results, columns=column_names)
columns_to_keep = ["dam_id", "name", "area_m2", "total_oper_cap"]
df_filtered = df[columns_to_keep]
print(df_filtered.head)

output_file= './csv/hospitals.csv'
df_filtered.to_csv(output_file, index=False)

<bound method NDFrame.head of     dam_id                          name       area_m2  total_oper_cap
0      159   NORTH UTAH CO - GROVE CREEK  1.242442e+07          1385.0
1      157  NORTH UTAH CO - BATTLE CREEK  1.447828e+07          1385.0
2       60                   QUAIL CREEK  2.086739e+07            80.0
3       61         QUAIL CREEK SOUTH DAM  2.076270e+07            80.0
4       93          DAVIS - BARTON CREEK  3.124641e+06            30.4
5        9           ROCKY FORD (BEAVER)  7.167679e+07             6.0
6      130                      RED PINE  2.517944e+06             4.8
7      204                    WHITE PINE  2.987596e+06             4.8
8       94       DAVIS - FARMINGTON POND  3.698298e+06             3.0
9       56            ENTERPRISE (LOWER)  1.502259e+08             3.0
10      57            ENTERPRISE (UPPER)  1.506308e+08             3.0
11      88                        MANTUA  2.837643e+06             1.8
12     172       LINDSAY (BENNETT) LOWER  4.403

Key fixes:

COALESCE("SUMMER_CAP", 0)::double precision in SQL

SUM(... )::double precision in SQL

float(...) in Python for safety

Guard if min_cap == max_cap to avoid a zero-width color scale

Make m the last line to render inline in the notebook

# Total Summer Operating Capacity

# Total Summer Operating Capacity, with Affected Population 

In [17]:
import pandas as pd
column_names = ["dam_id", "dam_num", "name", "geom_json", "beds", "num_hospitals", "trauma"]
df = pd.DataFrame(results, columns=column_names)
columns_to_keep = ["dam_id", "dam_num", "name", "beds", "num_hospitals", "trauma"]
df_filtered = df[columns_to_keep]
print(df_filtered.head)

<bound method NDFrame.head of     dam_id  dam_num                                       name  beds  \
0      169  UT00066                            CENTER CREEK #3    19   
1      161  UT00524                        PROVO - ROCK CANYON   395   
2      113  UT00832                    CEDAR CITY - DRY CANYON    48   
3      160  UT00299                NORTH UTAH CO - TIBBLE FORK    89   
4       93  UT00152                       DAVIS - BARTON CREEK    86   
5      128  UT00172                           LAKE MARY-PHOEBE    38   
6      186  UT00539                  SALT LAKE CO - SUGARHOUSE    14   
7      129  UT00755                                LITTLE DELL    14   
8       71  UT00221                              MOUNTAIN DELL    14   
9      162  UT00470                    PROVO - SLATE CANYON #2   384   
10     132  UT00531              SALT LAKE CO - CREEKSIDE PARK    38   
11      65  UT00039               KENNECOTT MINE BINGHAM CREEK   172   
12      82  UT00833         CEDAR 

In [18]:
output_file= './csv/hospital_dams.csv'
df_filtered.to_csv(output_file, index=False)